In [ ]:
import pandas as pd
occupations = pd.read_csv("data/acs_occupations_raw.csv", skiprows=4)

In [ ]:
occupations.head()

In [ ]:
# Rename entire header row to create the gender column

occupations.drop(occupations.index[2]).reset_index(drop=True)
occupations.rename(columns={"Unnamed: 0":"Gender"}, inplace=True)

occupations.ffill(axis=1)
headers = pd.Series(occupations.columns)
headers = headers.replace(r"Unnamed:.*", pd.NA, regex=True)
headers = headers.ffill()
occupations.columns = headers

occupations.head()

In [ ]:
# Transposed the data frame to make gender a column and set everything up for a .melt function

occupations_transposed = occupations.T
occupations_transposed = occupations_transposed.reset_index()
occupations_transposed.head(10)

In [ ]:
# Identify where the "total" rows are within the gender column (totals per gender)

occupations_transposed[
    occupations_transposed[0] == "Age recode (AGEP_RC1)"
]

# Drop the total rows (totals per gender)

occupations_transposed = occupations_transposed.drop(
    occupations_transposed[
        occupations_transposed[0] == "Age recode (AGEP_RC1)"
    ].index
).reset_index(drop=True)

occupations_transposed.head(10)

In [ ]:
# Drop the total row (population total)
total_rows = occupations_transposed[
    occupations_transposed[1] == "Total"
].index

occupations_transposed = occupations_transposed.drop(index=total_rows)

# Reset index
occupations_transposed = occupations_transposed.reset_index(drop=True)

In [ ]:
# Drop the column with all the total values
occupations_transposed = occupations_transposed.drop(
    columns=occupations_transposed.columns[[1, 3]]
)

# Reset column numbering
occupations_transposed.columns = range(
    len(occupations_transposed.columns)
)

In [ ]:
# Change column 2 to the "Age" column and set row 0 to the headers
occupations_transposed.iloc[0, 1] = "Age"
occupations_transposed.columns = occupations_transposed.iloc[0]
occupations_transposed = (occupations_transposed.iloc[1:].reset_index(drop=True))

In [ ]:
# Melt the data frame 

occupations_long = occupations_transposed.melt(
    id_vars=["Gender", "Age"],
    var_name="Occupation",
    value_name="Population"
)

# Validate final dataset
occupations_long.head(35)

In [ ]:
occupations_long.to_csv(
    "acs_occupations_clean.csv",
    index=False
)